# E52 FWB公开表：关系类型编码与比例有限核对

2026-10-30。仅核人数与编码对应，不复现回归、失访机制或整个研究。只保存聚合结果，原行与原工作簿不随仓库发布。

## 结论

公开表192行中190行有目前关系类别。按问卷语义映射后57行与原先希望一致：57/192=29.69%，57/190=30%。两种分母均四舍五入为正式更正的30%，因此不能仅靠舍入结果确定作者更正所用分母。希望恋人的47人中7人后来为恋人，希望朋友的22人中13人后来为朋友，约15%及59%。这是有限交叉计数核对。

## 来源与方法

项目 https://osf.io/pazmy/ ，文件目录 https://api.osf.io/v2/nodes/pazmy/files/osfstorage/ 。工作簿 https://osf.io/download/798m4/ （版本1，139474字节）；下载命名为 machia2020_data.xlsx，放在运行目录。需要Python、pandas、openpyxl。代码核对SHA-256，读取Sheet1，不改原件。

T1问卷 https://osf.io/download/fqzer/ 第4页：future，1保持FWB、2恋人、3只做朋友、4无关系、5其他。T2问卷 https://osf.io/download/e2dg9/ 第1页：t2relstat，1FWB、2朋友、3恋人、4无关系。两处2/3顺序互换，必须按定义映射。第5类“其他”没有预先明确目标，不把它强行配对或解释为失败。

### 关键限制

公开表没有稳定受访者ID列，零完全相同行不证明没有重复受访者。工作表记录范围335行含尾部空格式，pandas实际读得192行。只输出四分类交叉表，不输出个体记录或开放回答。完整论文方法及分析代码仍缺，未建立所有筛选口径，不作失访率或因果推断。

## 实际计算

下列代码已用普通Python顺序执行并保存输出，execution_count为null；未运行Jupyter内核，未作Notebook界面验收。可用安装了上述依赖及jupyter/nbconvert的环境运行 `python -m jupyter nbconvert --execute --to notebook --inplace E52_FWB公开表编码与比例有限核对.ipynb`。此小型人数核对用表格表达，未加图表。


In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
source = Path("machia2020_data.xlsx")
assert hashlib.sha256(source.read_bytes()).hexdigest() == "207424b595e3985c84e4851c000b160e97f45f39d0e10537a287f1f074c9da92"
data = pd.read_excel(source, sheet_name="Sheet1")
assert data.columns.is_unique
assert len(data) == 192
assert set(data["future"].dropna()) <= {1,2,3,4,5}
assert set(data["t2relstat"].dropna()) <= {1,2,3,4}
valid = data["t2relstat"].isin([1,2,3,4])
table = pd.crosstab(data.loc[valid,"future"],data.loc[valid,"t2relstat"])
mapping = {1:1, 2:3, 3:2, 4:4}
matched = (data["future"].map(mapping) == data["t2relstat"]).sum()
romance = data["future"].eq(2)
friends = data["future"].eq(3)
result = {
 "rows": len(data), "valid_current_type": int(valid.sum()),
 "missing_current_type": int(data["t2relstat"].isna().sum()),
 "full_row_duplicates": int(data.duplicated().sum()),
 "matched_desired_type": int(matched),
 "matched_over_all_rows_pct": float(matched/len(data)*100),
 "matched_over_valid_current_pct": float(matched/valid.sum()*100),
 "romance_desired_n": int(romance.sum()),
 "romance_desired_and_observed_n": int((romance & data["t2relstat"].eq(3)).sum()),
 "friendship_desired_n": int(friends.sum()),
 "friendship_desired_and_observed_n": int((friends & data["t2relstat"].eq(2)).sum()),
 "no_relationship_observed_n": int(data["t2relstat"].eq(4).sum()),
 "other_desired_n": int(data["future"].eq(5).sum())
}
assert matched == sum(int(table.loc[a,b]) for a,b in mapping.items())
print(json.dumps(result, indent=2, ensure_ascii=False))
print("T1希望编码（行）×T2目前编码（列）；只输出聚合人数")
print(table.to_string())


{
  "rows": 192,
  "valid_current_type": 190,
  "missing_current_type": 2,
  "full_row_duplicates": 0,
  "matched_desired_type": 57,
  "matched_over_all_rows_pct": 29.6875,
  "matched_over_valid_current_pct": 30.0,
  "romance_desired_n": 47,
  "romance_desired_and_observed_n": 7,
  "friendship_desired_n": 22,
  "friendship_desired_and_observed_n": 13,
  "no_relationship_observed_n": 59,
  "other_desired_n": 29
}
T1希望编码（行）×T2目前编码（列）；只输出聚合人数
t2relstat  1.0  2.0  3.0  4.0
future                       
1           34   16   11   24
2            9   13    7   18
3            2   13    1    6
4            0    2    2    3
5            4   10    7    8


## 解读与接续
同类别的期望与结果相符不代表关系质量或双方愿望一致。59名无关系者占190有效类别者31.05%，占192行30.73%，两者均约31%；不把一方报告等同双方确认。论文 DOI 10.1111/pere.12307；正式更正 DOI 10.1111/pere.12339（30%更正17%）。本次没有解释17%的具体产生原因，也未复现研究的模型。
